<div style = "background-color:#36A6A6; border-bottom:3px solid #FFD700; padding:20px; text-align:center;">
    <h1 style = "color:white; font-family:calibri; margin:0;"><b>Super Store Retail Profitability and Discount Uptimization Python Phase</b></h1>
    <h3 style = "color:#FFD700; font-family:Arial; margin:5px 0;">Data Analysis & Insights</h3>
    <p style = "color:white; font-style:italic; margin:0;">Powered by Jupyter Notebook</p>
</div>

### <span style="color:#36A6A6">Creating a safe SQLAlchemy engine for PostgreSQL</span>

In [1]:
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# IMPORTANT: For security I use environment variables
db_user = os.getenv("PGUSER", "postgres")
db_password = os.getenv("PGPASSWORD", "PAMPostG18")
db_host = os.getenv("PGHOST", "localhost")
db_port = os.getenv("PGPORT", "5432")
db_name = os.getenv("PGDB", "Super_StoreDB")

connection_url = URL.create(
    "postgresql+psycopg2",
    username = db_user,
    password = db_password,
    host = db_host,
    port = db_port,
    database = db_name
)

engine = create_engine(connection_url)
print("Engine created. Test connection below...")


# quick test
with engine.connect() as conn:
    r = conn.execute(text("SELECT 1")).scalar()
    print("Test query returned:", r)
    

Engine created. Test connection below...
Test query returned: 1


### <span style="color:#36A6A6">Loading CSV safely</span>

In [2]:
# Import pandas for data handling
import pandas as pd

# Load the uploaded CSV file
df = pd.read_csv("SuperStore_Dataset.csv", encoding="latin1")

# Confirm dataset loaded
print("Dataset loaded successfully ✅")
print("Shape of dataset:", df.shape)

# Preview first 5 rows
df.head()


Dataset loaded successfully ✅
Shape of dataset: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### <span style="color:#36A6A6">Cleaning and Standardizing Column Names</span>


In [3]:
# Remove leading/trailing spaces
df.columns = df.columns.str.strip()

# Convert to lowercase
df.columns = df.columns.str.lower()

# Replace spaces with underscore
df.columns = df.columns.str.replace(" ", "_")

# Display cleaned column names
print("Cleaned column names ✅")
print(df.columns)


Cleaned column names ✅
Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub-category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')


### <span style="color:#36A6A6">Checking Data Types and Converting Dates</span>


In [4]:
# Check current data types
print("Current Data Types:")
print(df.dtypes)


Current Data Types:
row_id             int64
order_id          object
order_date        object
ship_date         object
ship_mode         object
customer_id       object
customer_name     object
segment           object
country           object
city              object
state             object
postal_code        int64
region            object
product_id        object
category          object
sub-category      object
product_name      object
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object


### <span style="color:#36A6A6">Fixing Data Types for Database Readiness</span>


In [5]:
# Convert date columns
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')

# Convert postal_code to string (safer for database storage)
df['postal_code'] = df['postal_code'].astype(str)

print("Data type corrections completed ✅")
print("\nUpdated Data Types:")
print(df.dtypes)


Data type corrections completed ✅

Updated Data Types:
row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code              object
region                   object
product_id               object
category                 object
sub-category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object


### <span style="color:#36A6A6">Designing Star Schema Model</span>


In [6]:
# Create Customer Dimension
dim_customer = df[['customer_id', 'customer_name', 'segment']].drop_duplicates().reset_index(drop=True)

print("dim_customer shape:", dim_customer.shape)
dim_customer.head()


dim_customer shape: (793, 3)


,customer_id,customer_name,segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


In [7]:
# Create Product Dimension
dim_product = df[['product_id', 'category', 'sub-category', 'product_name']].drop_duplicates().reset_index(drop=True)

print("dim_product shape:", dim_product.shape)
dim_product.head()


dim_product shape: (1894, 4)


,product_id,category,sub-category,product_name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


In [8]:
# Create Location Dimension
dim_location = df[['country', 'city', 'state', 'postal_code', 'region']].drop_duplicates().reset_index(drop=True)

print("dim_location shape:", dim_location.shape)
dim_location.head()


dim_location shape: (632, 5)


,country,city,state,postal_code,region
0,United States,Henderson,Kentucky,42420,South
1,United States,Los Angeles,California,90036,West
2,United States,Fort Lauderdale,Florida,33311,South
3,United States,Los Angeles,California,90032,West
4,United States,Concord,North Carolina,28027,South


In [9]:
# Create Date Dimension
dim_date = df[['order_date']].drop_duplicates().reset_index(drop=True)

dim_date['year'] = dim_date['order_date'].dt.year
dim_date['month'] = dim_date['order_date'].dt.month
dim_date['month_name'] = dim_date['order_date'].dt.month_name()
dim_date['quarter'] = dim_date['order_date'].dt.quarter

print("dim_date shape:", dim_date.shape)
dim_date.head()


dim_date shape: (1237, 5)


,order_date,year,month,month_name,quarter
0,2016-11-08,2016,11,November,4
1,2016-06-12,2016,6,June,2
2,2015-10-11,2015,10,October,4
3,2014-06-09,2014,6,June,2
4,2017-04-15,2017,4,April,2


In [10]:
fact_sales = df[[
    'row_id',
    'order_id',
    'order_date',
    'customer_id',
    'product_id',
    'country',
    'city',
    'state',
    'postal_code',
    'sales',
    'quantity',
    'discount',
    'profit'
]].copy()

print("fact_sales shape:", fact_sales.shape)
fact_sales.head()


fact_sales shape: (9994, 13)


,row_id,order_id,order_date,customer_id,product_id,country,city,state,postal_code,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,CG-12520,FUR-BO-10001798,United States,Henderson,Kentucky,42420,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,CG-12520,FUR-CH-10000454,United States,Henderson,Kentucky,42420,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,DV-13045,OFF-LA-10000240,United States,Los Angeles,California,90036,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,SO-20335,FUR-TA-10000577,United States,Fort Lauderdale,Florida,33311,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,SO-20335,OFF-ST-10000760,United States,Fort Lauderdale,Florida,33311,22.3680,2,0.20,2.5164


### <span style="color:#36A6A6">Creating Star Schema Tables in PostgreSQL</span>


### <span style="color:#36A6A6">Adding surrogate keys first</span>


In [11]:
# Customer_Key

dim_customer.insert(0, 'customer_key', range(1, len(dim_customer) + 1))
dim_customer.head()


,customer_key,customer_id,customer_name,segment
0,1,CG-12520,Claire Gute,Consumer
1,2,DV-13045,Darrin Van Huff,Corporate
2,3,SO-20335,Sean O'Donnell,Consumer
3,4,BH-11710,Brosina Hoffman,Consumer
4,5,AA-10480,Andrew Allen,Consumer


In [12]:
# Product_Key

dim_product.insert(0, 'product_key', range(1, len(dim_product) + 1))
dim_product.head()


,product_key,product_id,category,sub-category,product_name
0,1,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,2,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,3,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,4,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,5,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


In [13]:
# Location_Key

dim_location.insert(0, 'location_key', range(1, len(dim_location) + 1))
dim_location.head()


,location_key,country,city,state,postal_code,region
0,1,United States,Henderson,Kentucky,42420,South
1,2,United States,Los Angeles,California,90036,West
2,3,United States,Fort Lauderdale,Florida,33311,South
3,4,United States,Los Angeles,California,90032,West
4,5,United States,Concord,North Carolina,28027,South


In [14]:
# Date_kay (I created the YYYYMMDD format)

dim_date['date_key'] = dim_date['order_date'].dt.strftime('%Y%m%d').astype(int)

# Move key to first column
cols = ['date_key'] + [col for col in dim_date.columns if col != 'date_key']
dim_date = dim_date[cols]

dim_date.head()


,date_key,order_date,year,month,month_name,quarter
0,20161108,2016-11-08,2016,11,November,4
1,20160612,2016-06-12,2016,6,June,2
2,20151011,2015-10-11,2015,10,October,4
3,20140609,2014-06-09,2014,6,June,2
4,20170415,2017-04-15,2017,4,April,2


### <span style="color:#36A6A6">Creating the SQL Database Tables with Python</span>


In [15]:
with engine.connect() as conn:

    # Drop tables if they exist (clean rebuild)
    conn.execute(text("DROP TABLE IF EXISTS fact_sales CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_customer CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_product CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_location CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_date CASCADE"))

# STEP 9 — Create table DDL (run once). Adjust schema name if needed.
ddl_statements = [
"""
 CREATE TABLE IF NOT EXISTS dim_customer (
            customer_key INT PRIMARY KEY,
            customer_id TEXT,
            customer_name TEXT,
            segment TEXT
);
""",   
"""
  CREATE TABLE IF NOT EXISTS dim_product (
            product_key INT PRIMARY KEY,
            product_id TEXT,
            category TEXT,
            sub_category TEXT,
            product_name TEXT
);
""",
"""
 CREATE TABLE IF NOT EXISTS dim_location (
            location_key INT PRIMARY KEY,
            country TEXT,
            city TEXT,
            state TEXT,
            postal_code TEXT,
            region TEXT
);
""",
"""
 CREATE TABLE IF NOT EXISTS dim_date (
            date_key INT PRIMARY KEY,
            order_date DATE,
            year INT,
            month INT,
            month_name TEXT,
            quarter INT
);
"""
]

with engine.begin() as conn:
    for s in ddl_statements:
        conn.exec_driver_sql(s)
print("DDL executed (Dimention tables created).✅")


DDL executed (Dimention tables created).✅


### <span style="color:#36A6A6">Mapping Foreign Keys into Fact Table</span>

In [16]:
# Map Customer Key
fact_sales = fact_sales.merge(
    dim_customer[['customer_key', 'customer_id']],
    on='customer_id',
    how='left'
)

In [17]:
# Map Product Key
fact_sales = fact_sales.merge(
    dim_product[['product_key', 'product_id']],
    on='product_id',
    how='left'
)

In [18]:
# map Location Key
fact_sales = fact_sales.merge(
    dim_location[['location_key', 'country', 'city', 'state', 'postal_code']],
    on=['country', 'city', 'state', 'postal_code'],
    how='left'
)

In [19]:
# map Date Key
fact_sales['date_key'] = fact_sales['order_date'].dt.strftime('%Y%m%d').astype(int)

In [20]:
print(fact_sales.columns)

Index(['row_id', 'order_id', 'order_date', 'customer_id', 'product_id',
       'country', 'city', 'state', 'postal_code', 'sales', 'quantity',
       'discount', 'profit', 'customer_key', 'product_key', 'location_key',
       'date_key'],
      dtype='object')


In [21]:
fact_sales = fact_sales[[
    'row_id',
    'order_id',
    'customer_key',
    'product_key',
    'location_key',
    'date_key',
    'sales',
    'quantity',
    'discount',
    'profit'
]]

In [22]:
fact_sales.head()

,row_id,order_id,customer_key,product_key,location_key,date_key,sales,quantity,discount,profit
0,1,CA-2016-152156,1,1,1,20161108,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,1,2,1,20161108,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2,3,2,20160612,14.6200,2,0.00,6.8714
3,4,US-2015-108966,3,4,3,20151011,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,3,5,3,20151011,22.3680,2,0.20,2.5164


### <span style="color:#36A6A6">Validate Foreign Keys Before Loading to PostgreSQL</span>

In [23]:
fact_sales[['customer_key','product_key','location_key','date_key']].isnull().sum()

customer_key    0
product_key     0
location_key    0
date_key        0
dtype: int64

### <span style="color:#36A6A6">Load Dimension Tables into PostgreSQL</span>

In [24]:
dim_customer.to_sql('dim_customer', engine, if_exists='replace', index=False)
dim_product.to_sql('dim_product', engine, if_exists='replace', index=False)
dim_location.to_sql('dim_location', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

237

### <span style="color:#36A6A6">Step 11: Add Primary Keys to Dimension Tables</span>

In [25]:
with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE dim_customer ADD PRIMARY KEY (customer_key);
        ALTER TABLE dim_product ADD PRIMARY KEY (product_key);
        ALTER TABLE dim_location ADD PRIMARY KEY (location_key);
        ALTER TABLE dim_date ADD PRIMARY KEY (date_key);
    """))
    conn.commit()

### <span style="color:#36A6A6">Step 12: Load Fact Table into PostgreSQL</span>

In [26]:
fact_sales.to_sql('fact_sales', engine, if_exists='replace', index=False)

331

In [27]:
fact_sales['row_id'].duplicated().sum()

337

In [28]:
len(fact_sales)

10331

In [29]:
print("Customer duplicates:", dim_customer['customer_id'].duplicated().sum())
print("Product duplicates:", dim_product['product_id'].duplicated().sum())
print("Location duplicates:",
      dim_location[['country','city','state','postal_code']].duplicated().sum())
print("Date duplicates:", dim_date['order_date'].duplicated().sum())

Customer duplicates: 0
Product duplicates: 32
Location duplicates: 0
Date duplicates: 0


In [30]:
dim_product = (
    df.sort_values('product_name')  # stable selection
      .drop_duplicates(subset=['product_id'])
      [['product_id','category','sub-category','product_name']]
      .reset_index(drop=True)
)

dim_product['product_key'] = dim_product.index + 1

In [31]:
dim_product['product_id'].duplicated().sum()

0

In [32]:
fact_sales.head()

,row_id,order_id,customer_key,product_key,location_key,date_key,sales,quantity,discount,profit
0,1,CA-2016-152156,1,1,1,20161108,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,1,2,1,20161108,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2,3,2,20160612,14.6200,2,0.00,6.8714
3,4,US-2015-108966,3,4,3,20151011,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,3,5,3,20151011,22.3680,2,0.20,2.5164


### <span style="color:#36A6A6">Creating Month_name abbreviation column</span>

In [33]:
engine = create_engine(connection_url)

# ============================
# Step 1: Add columns for Month_name abbreviation 
add_columns_sql = """
ALTER TABLE dim_date
ADD COLUMN IF NOT EXISTS month_abbr VARCHAR(3);
"""

# ============================
# Step 2: Populate the columns
update_columns_sql = """
UPDATE dim_date
SET
    month_abbr = LEFT(month_name, 3)
WHERE month_name IS NOT NULL;
"""

# ============================
# Execute safely
with engine.begin() as conn:
    conn.execute(text(add_columns_sql))
    conn.execute(text(update_columns_sql))

print("✅ dim_date updated with month_abbr columns successfully")

✅ dim_date updated with month_abbr columns successfully


### <span style = "color:#36A6A6;">Retrieving all tables from PostgreSQG</span>

In [34]:
import pandas as pd

for table in ["fact_sales","dim_date","dim_customer","dim_product","dim_location"]:
    print(f"\n--- {table.upper()} ---")
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))



--- FACT_SALES ---


,row_id,order_id,customer_key,product_key,location_key,date_key,sales,quantity,discount,profit
0,1,CA-2016-152156,1,1,1,20161108,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,1,2,1,20161108,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2,3,2,20160612,14.6200,2,0.00,6.8714
3,4,US-2015-108966,3,4,3,20151011,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,3,5,3,20151011,22.3680,2,0.20,2.5164



--- DIM_DATE ---


,date_key,order_date,year,month,month_name,quarter,month_abbr
0,20161108,2016-11-08,2016,11,November,4,Nov
1,20160612,2016-06-12,2016,6,June,2,Jun
2,20151011,2015-10-11,2015,10,October,4,Oct
3,20140609,2014-06-09,2014,6,June,2,Jun
4,20170415,2017-04-15,2017,4,April,2,Apr



--- DIM_CUSTOMER ---


,customer_key,customer_id,customer_name,segment
0,1,CG-12520,Claire Gute,Consumer
1,2,DV-13045,Darrin Van Huff,Corporate
2,3,SO-20335,Sean O'Donnell,Consumer
3,4,BH-11710,Brosina Hoffman,Consumer
4,5,AA-10480,Andrew Allen,Consumer



--- DIM_PRODUCT ---


,product_key,product_id,category,sub-category,product_name
0,1,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,2,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,3,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,4,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,5,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System



--- DIM_LOCATION ---


,location_key,country,city,state,postal_code,region
0,1,United States,Henderson,Kentucky,42420,South
1,2,United States,Los Angeles,California,90036,West
2,3,United States,Fort Lauderdale,Florida,33311,South
3,4,United States,Los Angeles,California,90032,West
4,5,United States,Concord,North Carolina,28027,South


### <span style="color:#36A6A6">Load Tables from PostgreSQL into Python</span>

In [36]:
import pandas as pd

fact = pd.read_sql("SELECT * FROM fact_sales", engine)
dim_customer = pd.read_sql("SELECT * FROM dim_customer", engine)
dim_product = pd.read_sql("SELECT * FROM dim_product", engine)
dim_date = pd.read_sql("SELECT * FROM dim_date", engine)
dim_location = pd.read_sql("SELECT * FROM dim_location", engine)

print("Tables loaded successfully.")

Tables loaded successfully.


### <span style="color:#36A6A6">Join Fact with Dimensions</span>

In [37]:
df_analysis = fact \
    .merge(dim_customer, on='customer_key') \
    .merge(dim_product, on='product_key') \
    .merge(dim_location, on='location_key') \
    .merge(dim_date, on='date_key')

print("Shape:", df_analysis.shape)

Shape: (10331, 28)


### <span style="color:#36A6A6">Getting the Min & Max Date</span>

In [2]:
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# IMPORTANT: For security I use environment variables
db_user = os.getenv("PGUSER", "postgres")
db_password = os.getenv("PGPASSWORD", "PAM@Postgres")
db_host = os.getenv("PGHOST", "localhost")
db_port = os.getenv("PGPORT", "5432")
db_name = os.getenv("PGDB", "Super_StoreDB")

connection_url = URL.create(
    "postgresql+psycopg2",
    username = db_user,
    password = db_password,
    host = db_host,
    port = db_port,
    database = db_name
)

engine = create_engine(connection_url)
print("Engine created. Test connection below...")

Engine created. Test connection below...


In [4]:
import pandas as pd
fact_sales = pd.read_sql("SELECT * FROM fact_sales", engine)

In [5]:
dim_date = pd.read_sql("SELECT * FROM dim_date", engine)

In [6]:
%who

URL	 connection_url	 create_engine	 db_host	 db_name	 db_password	 db_port	 db_user	 dim_date	 
engine	 fact_sales	 os	 pd	 text	 


In [7]:
dim_date.head()

,order_date,date_key,year,month,month_name,quarter,day,day_name
0,2014-01-03,20140103,2014,1,January,1,3,Friday
1,2014-01-04,20140104,2014,1,January,1,4,Saturday
2,2014-01-05,20140105,2014,1,January,1,5,Sunday
3,2014-01-06,20140106,2014,1,January,1,6,Monday
4,2014-01-07,20140107,2014,1,January,1,7,Tuesday


In [8]:
print("Min Date:", dim_date['order_date'].min())
print("Max Date:", dim_date['order_date'].max())

Min Date: 2014-01-03 00:00:00
Max Date: 2017-12-30 00:00:00


In [9]:
print(dim_date.columns)

Index(['order_date', 'date_key', 'year', 'month', 'month_name', 'quarter',
       'day', 'day_name'],
      dtype='object')


In [10]:
import pandas as pd
# Create full continuous date range
date_range = pd.date_range(
    start=dim_date['order_date'].min(),
    end=dim_date['order_date'].max(),
    freq='D'
)

# Build date dimension
dim_date = pd.DataFrame({
    'order_date': date_range
})

# Create date_key (YYYYMMDD format)
dim_date['date_key'] = dim_date['order_date'].dt.strftime('%Y%m%d').astype(int)

# Extract useful fields
dim_date['year'] = dim_date['order_date'].dt.year
dim_date['month'] = dim_date['order_date'].dt.month
dim_date['month_name'] = dim_date['order_date'].dt.month_name()
dim_date['quarter'] = dim_date['order_date'].dt.quarter
dim_date['day'] = dim_date['order_date'].dt.day
dim_date['day_name'] = dim_date['order_date'].dt.day_name()

dim_date.head()

,order_date,date_key,year,month,month_name,quarter,day,day_name
0,2014-01-03,20140103,2014,1,January,1,3,Friday
1,2014-01-04,20140104,2014,1,January,1,4,Saturday
2,2014-01-05,20140105,2014,1,January,1,5,Sunday
3,2014-01-06,20140106,2014,1,January,1,6,Monday
4,2014-01-07,20140107,2014,1,January,1,7,Tuesday


In [11]:
dim_date.to_sql(
    'dim_date',
    engine,
    if_exists='replace',
    index=False
)

458

### <span style="color:#36A6A6">Checking Duplicate row_id in fact_sales</span>

In [12]:
# Count duplicates
duplicate_count = fact_sales['row_id'].duplicated().sum()
print("Duplicate row_id count:", duplicate_count)

Duplicate row_id count: 337


### <span style="color:#36A6A6">Dropping Duplicate Records Based on row_id</span>

In [13]:
# Drop duplicates, keep first occurrence
fact_sales_clean = fact_sales.drop_duplicates(subset=['row_id'], keep='first')

print("Original rows:", len(fact_sales))
print("Rows after removing duplicates:", len(fact_sales_clean))

Original rows: 10331
Rows after removing duplicates: 9994


### <span style="color:#36A6A6">Verifying No Duplicates Remain</span>

In [14]:
print("Remaining duplicates:",
      fact_sales_clean['row_id'].duplicated().sum())

Remaining duplicates: 0


### <span style="color:#36A6A6">Uploading Clean fact_sales to PostgreSQL</span>

In [15]:
fact_sales_clean.to_sql(
    'fact_sales',
    engine,
    if_exists='replace',
    index=False
)

994

In [16]:
len(fact_sales_clean)

9994

### <span style="color:#36A6A6">Executive Summary Metrics</span>

In [38]:
total_sales = df_analysis['sales'].sum()
total_profit = df_analysis['profit'].sum()
profit_margin = total_profit / total_sales

print("Total Sales:", round(total_sales,2))
print("Total Profit:", round(total_profit,2))
print("Profit Margin:", round(profit_margin*100,2), "%")

Total Sales: 2394666.53
Total Profit: 299627.94
Profit Margin: 12.51 %


### <span style="color:#36A6A6">1️⃣ Understanding the Dataset Structure</span>

In [39]:
print("Shape of dataset:", df_analysis.shape)
df_analysis.head()

Shape of dataset: (10331, 28)


,row_id,order_id,customer_key,product_key,location_key,date_key,sales,quantity,discount,profit,...,city,state,postal_code,region,order_date,year,month,month_name,quarter,month_abbr
0,1,CA-2016-152156,1,1,1,20161108,261.9600,2,0.00,41.9136,...,Henderson,Kentucky,42420,South,2016-11-08,2016,11,November,4,Nov
1,2,CA-2016-152156,1,2,1,20161108,731.9400,3,0.00,219.5820,...,Henderson,Kentucky,42420,South,2016-11-08,2016,11,November,4,Nov
2,3,CA-2016-138688,2,3,2,20160612,14.6200,2,0.00,6.8714,...,Los Angeles,California,90036,West,2016-06-12,2016,6,June,2,Jun
3,4,US-2015-108966,3,4,3,20151011,957.5775,5,0.45,-383.0310,...,Fort Lauderdale,Florida,33311,South,2015-10-11,2015,10,October,4,Oct
4,5,US-2015-108966,3,5,3,20151011,22.3680,2,0.20,2.5164,...,Fort Lauderdale,Florida,33311,South,2015-10-11,2015,10,October,4,Oct


## <span style = "color:#36A6A6;">Some Real-World Business Analysis Questions Using Pandas Only (Python)</span>

### <span style="color:#36A6A6">2️⃣ Executive KPI Overview</span>

In [40]:
total_sales = df_analysis['sales'].sum()
total_profit = df_analysis['profit'].sum()
total_orders = df_analysis['order_id'].nunique()
total_customers = df_analysis['customer_id'].nunique()

print("Total Sales:", round(total_sales,2))
print("Total Profit:", round(total_profit,2))
print("Total Orders:", total_orders)
print("Total Customers:", total_customers)

Total Sales: 2394666.53
Total Profit: 299627.94
Total Orders: 5009
Total Customers: 793


### <span style="color:#36A6A6">3️⃣ Profitability Analysis</span>

#### <span style ="color:#36A6A6;">Answers to:</span>
#### <span style ="color:#36A6A6;">◆Which category makes the most profit?</span>
#### <span style ="color:#36A6A6;">◆Is any category losing money?</span>

In [41]:
# Profit by category
category_profit = df_analysis.groupby('category')[['sales','profit']].sum().sort_values(by='profit', ascending=False)

category_profit

,sales,profit
category,,
Technology,893633.2820,153415.7018
Office Supplies,736748.5940,126113.3459
Furniture,764284.6523,20098.8892


In [42]:
# Profit by subcategory
subcat_profit = df_analysis.groupby('sub-category')[['sales','profit']].sum().sort_values(by='profit')

subcat_profit

,sales,profit
sub-category,,
Tables,206965.5320,-17725.4811
Bookcases,127801.6393,-3452.8696
Supplies,46673.5380,-1189.0995
Fasteners,3024.2800,949.5182
Machines,194442.8720,2502.6381
Labels,12486.3120,5546.2540
Art,27199.7440,6551.5650
Envelopes,16476.4020,6964.1767
Furnishings,98626.3540,14569.5873


### <span style="color:#36A6A6">4️⃣ Discount Impact Analysis</span>

#### <span style ="color:#36A6A6;">Answers to:</span>
#### <span style ="color:#36A6A6;">◆At what discount level does the profit dropped?</span>
#### <span style ="color:#36A6A6;">◆Is high discount really helping?</span>

In [43]:
discount_analysis = df_analysis.groupby('discount')[['sales','profit']].mean().sort_index()

discount_analysis

,sales,profit
discount,,
0.00,227.398087,67.187087
0.10,576.346926,95.716963
0.15,573.393152,27.502370
0.20,213.391044,25.318906
0.30,447.547414,-45.034775
0.32,536.794770,-88.560656
0.40,567.293887,-112.479876
0.45,498.634000,-226.646464
0.50,892.705152,-310.703456


### <span style="color:#36A6A6">5️⃣ Customer Segment Analysis</span>

#### <span style ="color:#36A6A6;">Which Segment:</span>
#### <span style ="color:#36A6A6;">◆Buys more?</span>
#### <span style ="color:#36A6A6;">◆makes more Profit?</span>

In [44]:
segment_analysis = df_analysis.groupby('segment')[['sales','profit']].sum().sort_values(by='profit', ascending=False)

segment_analysis

,sales,profit
segment,,
Consumer,1.219188e+06,141152.8302
Corporate,7.260237e+05,94366.5195
Home Office,4.494546e+05,64108.5872


### <span style="color:#36A6A6">6️⃣ Regional Performance</span>

#### <span style ="color:#36A6A6;">To answer:</span>
#### <span style ="color:#36A6A6;">◆Which Region is the strongest?</span>
#### <span style ="color:#36A6A6;">◆Which Region needs attention?</span>

In [45]:
region_analysis = df_analysis.groupby('region')[['sales','profit']].sum().sort_values(by='profit', ascending=False)

region_analysis

,sales,profit
region,,
West,770331.1945,116437.8739
East,707955.6870,93963.7972
South,402534.2210,48606.2488
Central,513845.4258,40620.0170


### <span style="color:#36A6A6">7️⃣ Time Trend Analysis</span>

In [48]:
# Ensuring that the Order_date is Datetime
df_analysis['order_date'] = pd.to_datetime(df_analysis['order_date'])

df_analysis['year'] = df_analysis['order_date'].dt.year
df_analysis['month'] = df_analysis['order_date'].dt.month

In [47]:
# Yearly Trend
year_trend = df_analysis.groupby('year')[['sales','profit']].sum()

year_trend

,sales,profit
year,,
2014,502975.7291,51857.4857
2015,491333.3550,64835.3552
2016,637121.8880,85223.4449
2017,763235.5562,97711.6511


<div style = "background-color:#36A6A6; padding:20px; border-top:3px solid #FFD700;">
    <p style = "color:white; font-size:14px; margin:0;">Created by <b>Pam S. George</b> | Confidential Reporting</p>
</div>